In [ ]:
# Setup: garante que .env seja carregado e pacote esteja no path
import sys, os
from pathlib import Path

# Localiza raiz do projeto (sobe até encontrar pyproject.toml)
project_root = Path(os.getcwd())
for p in [project_root, *project_root.parents]:
    if (p / 'pyproject.toml').exists():
        project_root = p
        break

# Adiciona src/ ao sys.path (caso o pacote não esteja instalado em editable mode)
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Carrega .env da raiz do projeto
from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

print(f'Project root: {project_root}')
print(f'FRED_API_KEY: {"SET" if os.getenv("FRED_API_KEY") else "NOT SET — verifique o .env"}')

# Pipeline Completo — Carteira de Investimentos

Demonstra o pipeline end-to-end do carteira_auto:
**Carteira → Precos → Analise → Risco → Commodities → Rebalanceamento → Alertas**

Cobre:
1. Construir portfolio programaticamente
2. Executar pipeline completo via DAGEngine
3. Analisar resultados (metricas, risco, commodities)
4. Gerar recomendacoes de rebalanceamento
5. Avaliar alertas

**Nota**: Este notebook faz chamadas reais a APIs (Yahoo, BCB). Nao requer planilha Excel.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Construir Portfolio

Em producao, o portfolio e carregado da planilha Excel via `LoadPortfolioNode`.
Aqui construimos programaticamente para demonstracao.

In [ ]:
from carteira_auto.core.models import Asset, Portfolio
from carteira_auto.core.engine import PipelineContext

# Portfolio diversificado de exemplo
portfolio = Portfolio(assets=[
    # Acoes
    Asset(ticker="PETR4", nome="Petrobras PN", classe="Acoes",
          posicao_atual=8000, preco_atual=35.50, preco_medio=28.00,
          pct_meta=0.15, n_cotas_atual=225, proventos_recebidos=450),
    Asset(ticker="VALE3", nome="Vale ON", classe="Acoes",
          posicao_atual=6000, preco_atual=62.00, preco_medio=55.00,
          pct_meta=0.12, n_cotas_atual=97, proventos_recebidos=300),
    Asset(ticker="ITUB4", nome="Itau PN", classe="Acoes",
          posicao_atual=5000, preco_atual=32.00, preco_medio=30.00,
          pct_meta=0.10, n_cotas_atual=156, proventos_recebidos=200),
    Asset(ticker="WEGE3", nome="WEG ON", classe="Acoes",
          posicao_atual=4000, preco_atual=45.00, preco_medio=40.00,
          pct_meta=0.08, n_cotas_atual=89, proventos_recebidos=100),
    # Renda Fixa
    Asset(ticker="TESOURO_SELIC", nome="Tesouro Selic 2029", classe="Renda Fixa",
          posicao_atual=15000, preco_atual=15000, preco_medio=14000,
          pct_meta=0.30, n_cotas_atual=1),
    # FIIs
    Asset(ticker="HGLG11", nome="CSHG Logistica", classe="FIIs",
          posicao_atual=3000, preco_atual=165.00, preco_medio=160.00,
          pct_meta=0.10, n_cotas_atual=18, proventos_recebidos=180),
    Asset(ticker="XPML11", nome="XP Malls", classe="FIIs",
          posicao_atual=2500, preco_atual=110.00, preco_medio=105.00,
          pct_meta=0.08, n_cotas_atual=23, proventos_recebidos=150),
])

total = sum(a.posicao_atual for a in portfolio.assets)
print(f"Portfolio: {len(portfolio.assets)} ativos, total R${total:,.0f}")
for a in portfolio.assets:
    pct = a.posicao_atual / total * 100
    print(f"  {a.ticker:15s} {a.classe:12s} R${a.posicao_atual:>10,.0f} ({pct:.1f}%)")

In [ ]:
# Visualizar alocação do portfolio
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Por ativo
labels_ativos = [a.ticker for a in portfolio.assets]
valores_ativos = [a.posicao_atual for a in portfolio.assets]
cores_ativos = plt.cm.Set2(range(len(labels_ativos)))

ax1.pie(valores_ativos, labels=labels_ativos, colors=cores_ativos, autopct="%1.1f%%",
        startangle=90, pctdistance=0.85)
ax1.set_title("Alocação por Ativo")

# Por classe
from collections import defaultdict
por_classe = defaultdict(float)
for a in portfolio.assets:
    por_classe[a.classe] += a.posicao_atual

labels_classe = list(por_classe.keys())
valores_classe = list(por_classe.values())
cores_classe = {"Acoes": "#1f77b4", "Renda Fixa": "#2ca02c", "FIIs": "#ff7f0e"}
cores = [cores_classe.get(c, "#9467bd") for c in labels_classe]

ax2.pie(valores_classe, labels=labels_classe, colors=cores, autopct="%1.1f%%",
        startangle=90, pctdistance=0.85)
ax2.set_title("Alocação por Classe")

plt.suptitle(f"Portfolio — R$ {total:,.0f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Executar Analyzers Independentes

Analyzers sem dependencias (deps=[]) podem ser executados diretamente.
Cada um busca dados das APIs e produz metricas no contexto.

In [ ]:
from carteira_auto.analyzers import MacroAnalyzer, CommodityAnalyzer, FiscalAnalyzer

ctx = PipelineContext()
ctx["portfolio"] = portfolio

# Macro (BCB + IBGE)
macro = MacroAnalyzer()
ctx = macro.run(ctx)
mc = ctx["macro_context"]
print(f"Macro: Selic={mc.selic}%, IPCA={mc.ipca:.2f}%, PTAX=R${mc.cambio}")
print(f"  {mc.summary}")

In [ ]:
# Commodities (Yahoo)
commodity = CommodityAnalyzer()
ctx = commodity.run(ctx)
cm = ctx["commodity_metrics"]
print(f"Commodities: Brent=US${cm.oil_brent}, Ouro=US${cm.gold}")
print(f"  Ciclo: {cm.cycle_signal}")
print(f"  {cm.summary}")

In [ ]:
# Fiscal (BCB)
fiscal = FiscalAnalyzer()
ctx = fiscal.run(ctx)
fm = ctx["fiscal_metrics"]
print(f"Fiscal: divida={fm.divida_bruta_pib}% PIB, trajetoria={fm.fiscal_trajectory}")
print(f"  {fm.summary}")

In [ ]:
# Verificar erros parciais acumulados
if ctx.has_errors:
    print("\nErros parciais durante execucao:")
    for node, msg in ctx.errors.items():
        print(f"  {node}: {msg[:80]}..." if len(msg) > 80 else f"  {node}: {msg}")
else:
    print("\nNenhum erro parcial — todos os indicadores obtidos com sucesso")

## 3. Pipeline via Registry (CLI equivalente)

Em producao, pipelines sao executados via CLI:
```bash
python -m carteira_auto run macro
python -m carteira_auto run fiscal
python -m carteira_auto run commodities
```

Equivalente em Python:

In [ ]:
from carteira_auto.core.registry import create_engine, get_terminal_node, list_pipelines

# Listar todos os pipelines disponiveis
print(f"{len(list_pipelines())} pipelines disponiveis:")
for nome, descricao in sorted(list_pipelines().items()):
    print(f"  {nome:35s} | {descricao}")

In [ ]:
# Executar pipeline "fiscal" via engine
engine = create_engine()
terminal = get_terminal_node("fiscal")

# Dry-run primeiro
plan = engine.dry_run(terminal)
print(f"Pipeline 'fiscal' ({terminal}):")
print(f"  Plano: {' -> '.join(plan)}")

# Executar
ctx_fiscal = engine.run(terminal)
print(f"  Resultado: {ctx_fiscal['fiscal_metrics'].summary}")

## 4. MarketAnalyzer — Benchmarks

In [ ]:
from carteira_auto.analyzers import MarketAnalyzer

market = MarketAnalyzer()
ctx_market = PipelineContext()
ctx_market = market.run(ctx_market)

mm = ctx_market["market_metrics"]
print("=== Benchmarks de Mercado (acumulado 5 anos) ===")
print(f"IBOV:    {f'{mm.ibov_return:.1%}' if mm.ibov_return is not None else 'N/A'}")
print(f"IFIX:    {f'{mm.ifix_return:.1%}' if mm.ifix_return is not None else 'N/A'}")
print(f"CDI:     {f'{mm.cdi_return:.1%}' if mm.cdi_return is not None else 'N/A'}")
print(f"Selic:   {f'{mm.selic_retorno:.1%}' if mm.selic_retorno is not None else 'N/A'}")
print(f"S&P 500: {f'{mm.sp500_return:.1%}' if mm.sp500_return is not None else 'N/A'} (USD)")
print(f"Ouro:    {f'{mm.ouro_retorno:.1%}' if mm.ouro_retorno is not None else 'N/A'} (USD)")
print(f"Dólar:   {f'{mm.dolar_retorno:.1%}' if mm.dolar_retorno is not None else 'N/A'} (BRL/USD)")
print(f"PTAX:    R${mm.brl_usd}" if mm.brl_usd is not None else "PTAX:    N/A")

In [ ]:
# Gráfico: comparação de benchmarks (retorno acumulado 5 anos)
benchmarks = {
    "IBOV": mm.ibov_return,
    "IFIX": mm.ifix_return,
    "CDI": mm.cdi_return,
    "Selic": mm.selic_retorno,
    "S&P 500\n(USD)": mm.sp500_return,
    "Ouro\n(USD)": mm.ouro_retorno,
    "Dólar\n(BRL/USD)": mm.dolar_retorno,
}

# Filtrar apenas benchmarks com dados disponíveis
bench_labels = [k for k, v in benchmarks.items() if v is not None]
bench_vals = [v * 100 for k, v in benchmarks.items() if v is not None]  # converter para %

if bench_vals:
    cores_bench = ["#2ca02c" if v >= 0 else "#d62728" for v in bench_vals]
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(bench_labels, bench_vals, color=cores_bench, alpha=0.85, width=0.6)

    # Adicionar rótulos nas barras
    for bar, val in zip(bars, bench_vals):
        offset = 1.5 if val >= 0 else -3.5
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"{val:+.1f}%",
            ha="center", va="bottom" if val >= 0 else "top",
            fontsize=10, fontweight="bold"
        )

    ax.axhline(y=0, color="black", linewidth=0.8)
    ax.set_title("Benchmarks de Mercado — Retorno Acumulado 5 Anos", fontsize=14, fontweight="bold")
    ax.set_ylabel("Retorno Acumulado (%)")
    ax.tick_params(axis="x", labelsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Sem dados de benchmarks disponíveis para o gráfico")

print("=" * 60)
print("CENÁRIO CONSOLIDADO")
print("=" * 60)

print(f"\nMACRO BRASIL")
print(f"  Selic: {mc.selic}% | IPCA: {mc.ipca:.2f}%")
print(f"  PTAX: R${mc.cambio} | PIB: {mc.pib_growth}%")

print(f"\nFISCAL")
print(f"  Dívida/PIB: {fm.divida_bruta_pib}% | Primário: {fm.resultado_primario_pib}%")
print(f"  Trajetória: {fm.fiscal_trajectory}")

print(f"\nCOMMODITIES")
print(f"  Brent: US${cm.oil_brent} | Ouro: US${cm.gold}")
print(f"  Ciclo: {cm.cycle_signal}")

print(f"\nMERCADO (5 anos)")
ibov_str = f"{mm.ibov_return:.1%}" if mm.ibov_return is not None else "N/A"
ifix_str = f"{mm.ifix_return:.1%}" if mm.ifix_return is not None else "N/A"
cdi_str  = f"{mm.cdi_return:.1%}"  if mm.cdi_return  is not None else "N/A"
sp5_str  = f"{mm.sp500_return:.1%}" if mm.sp500_return is not None else "N/A"
print(f"  IBOV: {ibov_str} | IFIX: {ifix_str} | CDI: {cdi_str} | S&P500: {sp5_str}")

print(f"\nPORTFOLIO")
print(f"  {len(portfolio.assets)} ativos, R${total:,.0f}")
print("=" * 60)

In [ ]:
print("=" * 60)
print("CENÁRIO CONSOLIDADO")
print("=" * 60)

print(f"\nMACRO BRASIL")
print(f"  Selic: {mc.selic}% | IPCA: {mc.ipca:.2f}%")
print(f"  PTAX: R${mc.cambio} | PIB: {mc.pib_growth}%")

print(f"\nFISCAL")
print(f"  Dívida/PIB: {fm.divida_bruta_pib}% | Primário: {fm.resultado_primario_pib}%")
print(f"  Trajetória: {fm.fiscal_trajectory}")

print(f"\nCOMMODITIES")
print(f"  Brent: US${cm.oil_brent} | Ouro: US${cm.gold}")
print(f"  Ciclo: {cm.cycle_signal}")

print(f"\nMERCADO")
print(f"  IBOV: {mm.ibov} | IFIX: {mm.ifix}")

print(f"\nPORTFOLIO")
print(f"  {len(portfolio.assets)} ativos, R${total:,.0f}")
print("=" * 60)

## Resumo

Este notebook demonstrou o pipeline completo do carteira_auto:

1. **Portfolio** construido com Pydantic models validados
2. **Analyzers** executados independentemente (macro, fiscal, commodities, mercado)
3. **Registry** para executar pipelines nomeados (equivalente ao CLI)
4. **Error handling** parcial — falhas individuais nao interrompem a analise
5. **Visao consolidada** reunindo todos os contextos

Proximos passos (fases futuras):
- **Fase 3**: Estrategias de alocacao + otimizacao (PyPortfolioOpt)
- **Fase 4**: ML scoring fundamentalista
- **Fase 5**: NLP / sentimento / geopolitica
- **Fase 6**: AI Reasoning (Claude integrado)
- **Fase 7**: Publishers (Telegram, PDF, email)